In [91]:
# --------------------------
# Configuration & Path Setup
# --------------------------

import pandas as pd
import numpy as np
import os
from pathlib import Path
from typing import Union, Optional

# Cross-platform path resolution (consistent with Iteration 0)
def find_project_root_with_datasets(start_path: Path, max_levels: int = 10) -> Path:
    """
    Search upward from start_path for a directory containing 'Datasets' folder.
    This makes the notebook work on any system (macOS, Windows, Linux).
    """
    cur = start_path.resolve()
    for _ in range(max_levels):
        if (cur / 'Datasets').exists():
            return cur
        cur = cur.parent
    raise FileNotFoundError(
        f"Could not find project root with 'Datasets' folder within {max_levels} levels. "
        f"Set THESIS_BASE_DIR environment variable or ensure Datasets folder exists."
    )

# Try environment variable first, then search for project root
env_base = os.environ.get('THESIS_BASE_DIR')
if env_base:
    BASE_DIR = Path(env_base)
    print(f"Using THESIS_BASE_DIR from environment: {BASE_DIR}")
else:
    BASE_DIR = find_project_root_with_datasets(Path.cwd())
    print(f"Found project root: {BASE_DIR}")

PATHS = {
    'raw_data': BASE_DIR / "Datasets" / "RW_Datasets",
    'iteration_output': BASE_DIR / "Datasets" / "Iteration_Outputs" / "_iteration_1",
    'embeddings': BASE_DIR / "Datasets" / "Embeddings" / "_iteration_1" / "bert-base-cased",
    'results': BASE_DIR / "Results" / "_iteration_1"
}

# Normalize all paths to absolute Path objects
for k, p in list(PATHS.items()):
    PATHS[k] = Path(p).resolve()

# Create output directories
for path in PATHS.values():
    path.mkdir(parents=True, exist_ok=True)

# Data files configuration
DATA_FILES = {
    'df_14': PATHS['raw_data'] / "14k+ Reports RW_ACTUALS_PS.csv",
    'df_28': PATHS['raw_data'] / "28k+ Reports RW_ACTUALS_ALL.csv",
    'df_700': PATHS['raw_data'] / "700k+ Reports RW_ACTUALS_NH_OBS.csv"
}

def load_data(file_path: Union[str, Path], low_memory: bool = False) -> Optional[pd.DataFrame]:
    """
    Load CSV data into a DataFrame with error handling and encoding fallback.
    
    Parameters: 
        file_path (Union[str, Path]): Path to the CSV file
        low_memory (bool): Pass to pd.read_csv
        
    Returns:
        pandas.DataFrame: Loaded data or None if file not found
    """
    file_path = Path(file_path)  # Ensure it's a Path object
    
    # List of encodings to try
    encodings = ['utf-8', 'latin-1', 'iso-8859-1', 'cp1252']
    
    for encoding in encodings:
        try:
            df = pd.read_csv(file_path, low_memory=low_memory, encoding=encoding, on_bad_lines='skip')
            print(f"Successfully loaded {file_path.name} ({len(df)} rows) [encoding: {encoding}]")
            return df
        except (UnicodeDecodeError, Exception):
            if encoding == encodings[-1]:
                # Last attempt: try with error handling
                try:
                    df = pd.read_csv(file_path, low_memory=low_memory, encoding=encoding, 
                                    errors='ignore', on_bad_lines='skip')
                    print(f"Successfully loaded {file_path.name} ({len(df)} rows) [encoding: {encoding} with errors='ignore']")
                    return df
                except Exception as final_error:
                    print(f"An error occurred while loading {file_path.name}: {final_error}")
                    return None
            continue
    
    return None

# Load Each File
RW_ACTUALS_PS = load_data(DATA_FILES['df_14'])
RW_ACTUALS_ALL = load_data(DATA_FILES['df_28'])
RW_ACTUALS_NH_OBS = load_data(DATA_FILES['df_700'])

# Check if all dataframes were loaded successfully
if all(df is not None for df in [RW_ACTUALS_PS, RW_ACTUALS_ALL, RW_ACTUALS_NH_OBS]):
    print("\n All files loaded successfully!")
else:
    print("\n Warning: Some files failed to load. Please check the file paths and try again.")

Found project root: /Users/shariarimrozekhan/Documents/GitHub/masterThesis2026
Successfully loaded 14k+ Reports RW_ACTUALS_PS.csv (14432 rows) [encoding: utf-8]
Successfully loaded 28k+ Reports RW_ACTUALS_ALL.csv (28323 rows) [encoding: utf-8]
Successfully loaded 700k+ Reports RW_ACTUALS_NH_OBS.csv (1069414 rows) [encoding: latin-1]

 All files loaded successfully!
